# Fase 3 — Enriquecimento da camada Gold

Objetivo: adicionar informação de negócio que não existe na Silver — área do
parque (a partir da câmera) e período do dia (a partir do horário) — e deixar
uma tabela pronta para as análises da Fase 4.

In [0]:
spark.sql("USE CATALOG fauna")
spark.sql("USE SCHEMA monitoramento")

df_silver = spark.table("fauna.monitoramento.silver_registros")
print(f"Lendo silver_registros: {df_silver.count()} registros")

## Criar tabela de dimensão: câmera → área

In [0]:
dados_camera = [
    ("CAM01", "Cerrado aberto"), ("CAM02", "Cerrado aberto"),
    ("CAM03", "Cerrado aberto"), ("CAM04", "Cerrado aberto"),
    ("CAM05", "Mata de galeria"), ("CAM06", "Mata de galeria"),
    ("CAM07", "Mata de galeria"), ("CAM08", "Mata de galeria"),
    ("CAM09", "Veredas e áreas úmidas"), ("CAM10", "Veredas e áreas úmidas"),
    ("CAM11", "Veredas e áreas úmidas"), ("CAM12", "Veredas e áreas úmidas"),
]

df_dim_camera = spark.createDataFrame(dados_camera, ["id_camera", "area"])

(
    df_dim_camera.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fauna.monitoramento.dim_camera")
)

display(spark.table("fauna.monitoramento.dim_camera"))

## Enriquecer: área + período do dia

In [0]:
from pyspark.sql import functions as F

In [0]:
df_gold = (
    df_silver
    .join(df_dim_camera, on="id_camera", how="left")
    .withColumn("hora", F.hour("data_hora_inicio"))
    .withColumn(
        "periodo_dia",
        F.when((F.col("hora") >= 0) & (F.col("hora") <= 5), "Madrugada")
         .when((F.col("hora") >= 6) & (F.col("hora") <= 11), "Manhã")
         .when((F.col("hora") >= 12) & (F.col("hora") <= 17), "Tarde")
         .otherwise("Noite")
    )
)

df_gold.select("id_camera", "area", "data_hora_inicio", "hora", "periodo_dia").show(10, truncate=False)

## Confirmar integridade do join

In [0]:
nulos_area = df_gold.filter(F.col("area").isNull()).count()
print(f"Registros sem área após o join: {nulos_area}")

## Conferir distribuição real de todos os períodos

In [0]:
df_gold.groupBy("periodo_dia").count().orderBy(F.desc("count")).show()

## Gravar tabela Gold

In [0]:
TABELA_GOLD = "fauna.monitoramento.gold_registros"

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD)
)

print(f"✅ Tabela criada: {TABELA_GOLD}")

## Validação final da Gold

In [0]:
df_check = spark.table("fauna.monitoramento.gold_registros")

print(f"Total de linhas: {df_check.count()}")
print(f"Total de colunas: {len(df_check.columns)}")

print("\nDistribuição por área:")
df_check.groupBy("area").count().orderBy(F.desc("count")).show(truncate=False)

## Resumo e decisões da Fase 3 — Enriquecimento Gold

Esta fase juntou a Silver com informação de negócio que ainda não existia:
área do parque (a partir da câmera) e período do dia (a partir do horário).

**Decisões tomadas e por quê:**

- **`dim_camera` como tabela separada, não um dicionário no código.** A relação
  câmera → área virou uma tabela de dimensão versionada no catálogo
  (`fauna.monitoramento.dim_camera`), reaproveitável por qualquer notebook
  futuro — em vez de uma constante escondida dentro de um script.

- **`left join` em vez de `inner join`.** Mesmo sem nenhuma câmera fora do
  domínio confirmada na Fase 2, o `left join` garante que uma câmera nova ou
  não mapeada apareceria com `area = null` (visível, investigável) em vez de
  desaparecer silenciosamente do resultado.

- **Convenção explícita de período do dia**, documentada para não repetir a
  ambiguidade vista no rascunho gerado pelo assistente (que usava um `pd.cut`
  sem deixar claro o que acontece exatamente nos limites): Madrugada = 00h–05h59,
  Manhã = 06h–11h59, Tarde = 12h–17h59, Noite = 18h–23h59. Cada hora cheia
  pertence sempre ao período que ela inicia.

**Resultado:** tabela `fauna.monitoramento.gold_registros`, 5.000 linhas
(nenhuma perdida ou duplicada pelo join), 22 colunas. Distribuição por área:
Veredas e áreas úmidas (1.817), Cerrado aberto (1.624), Mata de galeria (1.559).
Distribuição por período do dia: Manhã (1.419), Tarde (1.317), Noite (1.177),
Madrugada (1.087).

**Próximo passo (Fase 4):** responder aos 5 desafios principais consultando
`gold_registros`.